# Order Management with ib-engine

Submit, monitor, and cancel orders via the Python API.

**Requirements:** `.env` with credentials. Market hours (9:30-16:00 ET) for DAY orders.

In [1]:
import os
import time
from dotenv import load_dotenv
import ibx

load_dotenv()

print("Connecting...")
engine = ibx.connect(
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
print(f"Connected! Account: {engine.account_id}")

Connecting...


Connected! Account: DUXXXXXXX


In [2]:
# Subscribe to SPY
spy = engine.subscribe(conid=756733, symbol="SPY")
time.sleep(3)  # wait for market data

quote = engine.quote(spy)
print(f"SPY: bid=${quote.bid:.2f} ask=${quote.ask:.2f}")

if quote.bid == 0:
    print("WARNING: No market data. Market may be closed.")

SPY: bid=$2043.79 ask=$3406.52


In [3]:
# Submit a limit BUY order $5 below the bid (won't fill immediately)
limit_price = round(quote.bid - 5.0, 2)
order_id = engine.submit_limit(spy, "BUY", qty=1, price=limit_price)
print(f"Submitted limit BUY 1 SPY @ ${limit_price:.2f} (order_id={order_id})")

# Wait for acknowledgement
time.sleep(2)
updates = engine.order_updates()
for u in updates:
    print(f"  -> {u}")

Submitted limit BUY 1 SPY @ $2038.79 (order_id=1773132404000)


  -> OrderUpdate(order_id=1773132404000, status=Submitted)


In [4]:
# Modify the order - move price $1 closer
new_price = round(limit_price + 1.0, 2)
new_order_id = engine.modify(order_id, price=new_price, qty=1)
print(f"Modified order {order_id} -> {new_order_id} @ ${new_price:.2f}")

time.sleep(2)
updates = engine.order_updates()
for u in updates:
    print(f"  -> {u}")

Modified order 1773132404000 -> 1773132404001 @ $2039.79


  -> OrderUpdate(order_id=1773132404001, status=Submitted)


In [5]:
# Cancel the order
engine.cancel(new_order_id)
print(f"Cancel requested for order {new_order_id}")

time.sleep(2)
updates = engine.order_updates()
for u in updates:
    print(f"  -> {u}")

# Check for any fills (should be empty)
fills = engine.fills()
print(f"\nFills: {len(fills)}")
for f in fills:
    print(f"  -> {f}")

Cancel requested for order 1773132404001


  -> OrderUpdate(order_id=1773132404001, status=Cancelled)

Fills: 0


In [6]:
# Cleanup
engine.shutdown()
print("Engine stopped.")

Engine stopped.
